# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR^2 dataset (colorectal cancer survivors) using the [`mlcroissant`](https://mlcroissant.readthedocs.io/en/latest/) library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset overview
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs (referenced by `@id`).

In [ ]:
# List all available record sets in the dataset via their @id

print("Available record sets (by @id and name):")
record_sets = [r for r in dataset._metadata['recordSet']] if hasattr(dataset, '_metadata') and 'recordSet' in dataset._metadata else []
if not record_sets:
    # Try the top-level 'recordSet' attribute
    from mlcroissant._dataset.metadata.dataset import RecordSet
    try:
        record_sets = [r['@id'] for r in dataset.metadata._jsonld.get('recordSet', [])]
    except Exception:
        record_sets = []
    if not record_sets:
        print(" - No record sets found. The schema may not include explicit record sets.")
else:
    for rs in record_sets:
        print(f" - {rs.get('@id', 'unknown id')} (name: {rs.get('name', 'unknown')})")

# Try using dataset.records() to discover available record sets (will error if none)
try:
    record_sets_detected = list(dataset.record_set_ids())
    if record_sets_detected:
        print("\nRecord sets detected from schema:")
    for rs_id in record_sets_detected:
        print(f" - {rs_id}")
except Exception as e:
    print("Failed to list record set IDs via dataset.record_set_ids().")
    record_sets_detected = []

# For this dataset, we will use the first available record set if present, otherwise print a warning.
selected_record_set_id = None
if record_sets_detected:
    selected_record_set_id = record_sets_detected[0]
    print(f"\nUsing record set: {selected_record_set_id}")
else:
    print("No record set found. Please examine the dataset metadata for available records.")

In [ ]:
# Explore a sample of records from the first available record set (using @id as required)
# Since mlcroissant guarantees usage by @id, pass selected_record_set_id as a variable
if selected_record_set_id:
    for i, record in enumerate(dataset.records(record_set=selected_record_set_id)):
        if i >= 3:
            break
        print(record)
else:
    print("No record set available for browsing records.")

## 3. Data Extraction
Load data from one or more specific record sets into a DataFrame for analysis. Refer to record set and field `@id` values above.

In [ ]:
# Gather all available record set @ids
try:
    record_set_ids = list(dataset.record_set_ids())
except Exception:
    record_set_ids = []
# If none detected, print a warning
if not record_set_ids:
    print("No record sets detected in dataset. Extraction cannot proceed.")
    dataframes = {}
else:
    dataframes = {}
    for rs_id in record_set_ids:
        print(f"Extracting records from record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

    # Show columns of the first record set (by @id)
    sample_rs = record_set_ids[0]
    print(f"\nColumns in DataFrame for record set {sample_rs}:")
    print(dataframes[sample_rs].columns.tolist())
    print("\nData sample:")
    display(dataframes[sample_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using field and record set `@id`s, e.g. filtering, normalization, grouping.

Suppose we select a numeric field from the dataset; we first need to display all columns for inspection using their `@id`.

In [ ]:
# Identify candidate numeric fields in the sample record set
sample_rs = record_set_ids[0] if record_set_ids else None
if sample_rs and not dataframes[sample_rs].empty:
    print("Sample data columns (first 5 rows):")
    display(dataframes[sample_rs].head())
    print("\nAvailable fields:")
    for i, col in enumerate(dataframes[sample_rs].columns):
        print(f"{i}: {col}")

    # Suppose 'schema:age' is available as a numeric field, or try to auto-select any numeric column
    numeric_candidates = dataframes[sample_rs].select_dtypes(include='number').columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"\nUsing numeric field for EDA: {numeric_field_id}")
    else:
        print("No numeric fields detected, EDA may not be possible.")
        numeric_field_id = None
    
    # Pick a group field, e.g. 'schema:sex' or first object/categorical column
    group_candidates = dataframes[sample_rs].select_dtypes(include='object').columns.tolist()
    group_field_id = None
    for gf in group_candidates:
        if "sex" in gf.lower() or "gender" in gf.lower() or "location" in gf.lower() or "group" in gf.lower():
            group_field_id = gf
            break
    if not group_field_id and group_candidates:
        group_field_id = group_candidates[0]
    print(f"Using group field: {group_field_id}")
else:
    print("No data available for EDA.")

In [ ]:
# Perform EDA: Filtering, normalization, and grouping
if sample_rs and numeric_field_id and not dataframes[sample_rs].empty:
    df = dataframes[sample_rs]

    # Example threshold for filtering; set threshold as mean - std
    threshold = df[numeric_field_id].mean() - df[numeric_field_id].std()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records in record set {sample_rs} with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by the group_field, if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        display(grouped_df.head())
    else:
        print("No valid group field for grouping.")
else:
    print("No numeric field or data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields (referenced by their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field and group means
if sample_rs and numeric_field_id and not dataframes[sample_rs].empty:
    df = dataframes[sample_rs]
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Barplot by group field if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(7,4))
        order = df[group_field_id].value_counts().index
        sns.barplot(x=group_field_id, y=numeric_field_id, data=df, order=order)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, preprocess, and visualize a biomedical dataset described by a Croissant schema using the `mlcroissant` library. All entities (record sets, fields, columns) were referenced only by their `@id` for clarity and reproducibility. This approach enables transparent and FAIR data exploration workflows tailored for clinical informatics and reproducible data analysis.